In [1]:
# 1. Import thư viện xử lý dữ liệu
import pandas as pd
import numpy as np

# 2. Import công cụ xử lý đường dẫn
from pathlib import Path

# 3. Import mô hình Decision Tree
from sklearn.tree import DecisionTreeClassifier

# 4. Import các công cụ đánh giá mô hình
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# 5. Import công cụ lưu mô hình
import joblib

# 6. Xác định thư mục gốc project
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data" / "processed" / "train.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

# 7. Xác định thư mục dữ liệu
DATA_DIR = PROJECT_DIR / "data" / "processed"

# 8. Đọc dữ liệu đã chia tập
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# 9. Kiểm tra kích thước
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (315125, 5)
Validation: (67524, 5)
Test: (67524, 5)


In [2]:
# 1. Danh sách 10 đặc trưng sử dụng cho mô hình
FEATURE_COLUMNS = [
    "url_length",
    "hostname_length",
    "dot_count",
    "hyphen_count",
    "digit_count",
    "at_count",
    "slash_count",
    "question_count",
    "equal_count",
    "has_https"
]

# 2. Tên cột nhãn cần dự đoán
TARGET_COLUMN = "type"


# 3. Hàm trích xuất đặc trưng từ URL
def extract_features(data):

    features = pd.DataFrame(index=data.index)

    # Độ dài URL
    features["url_length"] = data["normalized_url"].str.len()

    # Độ dài hostname
    features["hostname_length"] = data["hostname"].str.len()

    # Số dấu chấm
    features["dot_count"] = data["normalized_url"].str.count(r"\.")

    # Số dấu gạch ngang
    features["hyphen_count"] = data["normalized_url"].str.count("-")

    # Số chữ số
    features["digit_count"] = data["normalized_url"].str.count(r"\d")

    # Số ký tự @
    features["at_count"] = data["normalized_url"].str.count("@")

    # Số dấu /
    features["slash_count"] = data["normalized_url"].str.count("/")

    # Số dấu ?
    features["question_count"] = data["normalized_url"].str.count(r"\?")

    # Số dấu =
    features["equal_count"] = data["normalized_url"].str.count("=")

    # URL có sử dụng HTTPS hay không
    features["has_https"] = (
        data["normalized_url"]
        .str.lower()
        .str.startswith("https://")
        .astype(int)
    )

    return features

In [3]:
# 1. Trích xuất đặc trưng từ tập Train
train_features = extract_features(train_df)

# 2. Trích xuất đặc trưng từ tập Validation
val_features = extract_features(val_df)

# 3. Trích xuất đặc trưng từ tập Test
test_features = extract_features(test_df)

# 4. Tách dữ liệu đầu vào X và nhãn cần dự đoán y
X_train = train_features[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_val = val_features[FEATURE_COLUMNS]
y_val = val_df[TARGET_COLUMN]

X_test = test_features[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

# 5. Kiểm tra kích thước dữ liệu
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (315125, 10)
y_train: (315125,)
X_val: (67524, 10)
y_val: (67524,)
X_test: (67524, 10)
y_test: (67524,)


In [4]:
# 1. Khởi tạo mô hình Decision Tree
dt_model = DecisionTreeClassifier(
    random_state=42
)

# 2. Huấn luyện mô hình bằng tập Train
dt_model.fit(X_train, y_train)

# 3. Thông báo kết quả
print("Huấn luyện Decision Tree hoàn tất!")

# 4. Kiểm tra số lượng nút và độ sâu của cây
print("Độ sâu của cây:", dt_model.get_depth())
print("Số lượng nút:", dt_model.tree_.node_count)

Huấn luyện Decision Tree hoàn tất!
Độ sâu của cây: 38
Số lượng nút: 8915


In [5]:
# 1. Dự đoán nhãn trên tập Validation
y_val_pred = dt_model.predict(X_val)

# 2. So sánh một số nhãn thực tế và nhãn dự đoán
comparison_df = pd.DataFrame({
    "Nhãn thực tế": y_val.values,
    "Nhãn dự đoán": y_val_pred
})

# 3. Hiển thị 10 kết quả đầu tiên
display(comparison_df.head(10))

# 4. Đếm số lượng dự đoán đúng và sai
correct = (y_val.values == y_val_pred).sum()
incorrect = (y_val.values != y_val_pred).sum()

print("Số dự đoán đúng:", correct)
print("Số dự đoán sai:", incorrect)
print("Tổng số mẫu:", len(y_val))

,Nhãn thực tế,Nhãn dự đoán
0,legitimate,legitimate
1,legitimate,legitimate
2,legitimate,legitimate
3,legitimate,legitimate
4,legitimate,legitimate
5,legitimate,legitimate
6,legitimate,legitimate
7,legitimate,legitimate
8,legitimate,legitimate
9,legitimate,legitimate


Số dự đoán đúng: 66538
Số dự đoán sai: 986
Tổng số mẫu: 67524


In [6]:
# 1. Tính Accuracy
val_accuracy = accuracy_score(y_val, y_val_pred)

# 2. Tính Precision cho lớp phishing
val_precision = precision_score(
    y_val,
    y_val_pred,
    pos_label="phishing",
    zero_division=0
)

# 3. Tính Recall cho lớp phishing
val_recall = recall_score(
    y_val,
    y_val_pred,
    pos_label="phishing",
    zero_division=0
)

# 4. Tính F1-score cho lớp phishing
val_f1 = f1_score(
    y_val,
    y_val_pred,
    pos_label="phishing",
    zero_division=0
)

# 5. Hiển thị kết quả
print("===== KẾT QUẢ ĐÁNH GIÁ DECISION TREE =====")
print(f"Accuracy : {val_accuracy:.4f}")
print(f"Precision: {val_precision:.4f}")
print(f"Recall   : {val_recall:.4f}")
print(f"F1-score : {val_f1:.4f}")

===== KẾT QUẢ ĐÁNH GIÁ DECISION TREE =====
Accuracy : 0.9854
Precision: 0.9636
Recall   : 0.9739
F1-score : 0.9687


In [7]:
# 1. Tạo ma trận nhầm lẫn
cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=["legitimate", "phishing"]
)

# 2. Hiển thị ma trận
print("===== CONFUSION MATRIX =====")
print(cm)

# 3. Tách các giá trị trong ma trận
tn, fp, fn, tp = cm.ravel()

print("\n===== CHI TIẾT =====")
print("True Negative (TN):", tn)
print("False Positive (FP):", fp)
print("False Negative (FN):", fn)
print("True Positive (TP):", tp)

===== CONFUSION MATRIX =====
[[51284   577]
 [  409 15254]]

===== CHI TIẾT =====
True Negative (TN): 51284
False Positive (FP): 577
False Negative (FN): 409
True Positive (TP): 15254


In [8]:
# 1. Dự đoán trên tập Train
y_train_pred = dt_model.predict(X_train)

# 2. Tính Accuracy trên Train
train_accuracy = accuracy_score(y_train, y_train_pred)

# 3. Tính Precision cho lớp phishing
train_precision = precision_score(
    y_train,
    y_train_pred,
    pos_label="phishing",
    zero_division=0
)

# 4. Tính Recall cho lớp phishing
train_recall = recall_score(
    y_train,
    y_train_pred,
    pos_label="phishing",
    zero_division=0
)

# 5. Tính F1-score cho lớp phishing
train_f1 = f1_score(
    y_train,
    y_train_pred,
    pos_label="phishing",
    zero_division=0
)

# 6. Tạo bảng so sánh
comparison_metrics = pd.DataFrame({
    "Chỉ số": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Train": [
        train_accuracy,
        train_precision,
        train_recall,
        train_f1
    ],
    "Validation": [
        val_accuracy,
        val_precision,
        val_recall,
        val_f1
    ]
})

# 7. Hiển thị kết quả
display(comparison_metrics.round(4))

,Chỉ số,Train,Validation
0,Accuracy,0.9975,0.9854
1,Precision,0.9997,0.9636
2,Recall,0.9897,0.9739
3,F1-score,0.9947,0.9687


In [9]:
# 1. Tạo báo cáo phân loại
report = classification_report(
    y_val,
    y_val_pred,
    labels=["legitimate", "phishing"],
    target_names=["legitimate", "phishing"],
    zero_division=0
)

# 2. Hiển thị báo cáo
print("===== CLASSIFICATION REPORT =====")
print(report)

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

  legitimate       0.99      0.99      0.99     51861
    phishing       0.96      0.97      0.97     15663

    accuracy                           0.99     67524
   macro avg       0.98      0.98      0.98     67524
weighted avg       0.99      0.99      0.99     67524



In [10]:
# 1. Xác định thư mục lưu mô hình
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

# 2. Tạo thư mục nếu chưa tồn tại
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Xác định đường dẫn lưu mô hình
MODEL_PATH = ARTIFACT_DIR / "decision_tree_baseline.joblib"

# 4. Lưu mô hình
joblib.dump(dt_model, MODEL_PATH)

# 5. Kiểm tra kết quả
print("Đã lưu mô hình thành công!")
print("Đường dẫn:", MODEL_PATH)
print("Kích thước file:", round(MODEL_PATH.stat().st_size / 1024, 2), "KB")

Đã lưu mô hình thành công!
Đường dẫn: D:\HocTap\Machine_learning\phishing-detection\artifacts\decision_tree_baseline.joblib
Kích thước file: 698.26 KB


In [11]:
import json
import pandas as pd

from pathlib import Path

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [12]:
import json
from pathlib import Path

# Xác định thư mục gốc dự án an toàn khi chạy từ notebooks hoặc project root
ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "train.csv").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "processed"
FEATURES = json.loads((ROOT / "artifacts" / "features.json").read_text(encoding="utf-8"))

# Dùng lại dữ liệu đã nạp ở các ô đầu, tránh tạo bản sao lớn trong RAM
train = train_df
val = val_df
y_train = train[TARGET_COLUMN]
y_val = val[TARGET_COLUMN]

def make_features(frame):
    url = frame["normalized_url"].fillna("").astype(str)
    host = frame["hostname"].fillna("").astype(str)
    return pd.DataFrame({
        "url_length": url.str.len(),
        "hostname_length": host.str.len(),
        "dot_count": url.str.count(r"\."),
        "hyphen_count": url.str.count("-"),
        "digit_count": url.str.count(r"\d"),
        "at_count": url.str.count("@"),
        "slash_count": url.str.count("/"),
        "question_count": url.str.count(r"\?"),
        "equal_count": url.str.count("="),
        "has_https": url.str.lower().str.startswith("https://").astype("int8"),
    }, index=frame.index)[FEATURES]

x_train = make_features(train)
x_val = make_features(val)
print("ROOT:", ROOT)
print("Train:", x_train.shape)
print("Validation:", x_val.shape)
print("Features:", FEATURES)


ROOT: D:\HocTap\Machine_learning\phishing-detection
Train: (315125, 10)
Validation: (67524, 10)
Features: ['url_length', 'hostname_length', 'dot_count', 'hyphen_count', 'digit_count', 'at_count', 'slash_count', 'question_count', 'equal_count', 'has_https']


In [13]:
# 1. Khởi tạo Decision Tree với độ sâu tối đa bằng 10
model = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

# 2. Huấn luyện bằng dữ liệu Train
model.fit(x_train, y_train)

# 3. Dự đoán trên Validation
y_pred = model.predict(x_val)

# 4. Tính các chỉ số
accuracy = accuracy_score(y_val, y_pred)

precision = precision_score(
    y_val, y_pred,
    pos_label="phishing"
)

recall = recall_score(
    y_val, y_pred,
    pos_label="phishing"
)

f1 = f1_score(
    y_val, y_pred,
    pos_label="phishing"
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.9912327468751851
Precision: 0.9960502929366072
Recall: 0.9660346038434527
F1: 0.9808128605691321
Tree depth: 10


In [14]:
# EXP003: Decision Tree max_depth = 15

model = DecisionTreeClassifier(
    max_depth=15,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.9915881760559209
Precision: 0.9939786635250998
Recall: 0.9696099087020367
F1: 0.9816430741387111
Tree depth: 15


In [15]:
# EXP004: Decision Tree max_depth = 20

model = DecisionTreeClassifier(
    max_depth=20,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.9907736508500681
Precision: 0.9879948085658663
Recall: 0.9720360084275043
F1: 0.979950439288128
Tree depth: 20


In [16]:
# EXP005: Decision Tree min_samples_split = 10

model = DecisionTreeClassifier(
    min_samples_split=10,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.9902553166281618
Precision: 0.9853787927799702
Recall: 0.9724190768052098
F1: 0.9788560411311054
Tree depth: 38


In [17]:
# EXP006: Decision Tree min_samples_split = 50

model = DecisionTreeClassifier(
    min_samples_split=50,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.9905811267105029
Precision: 0.9884287850224274
Recall: 0.9707591138351529
F1: 0.9795142691490047
Tree depth: 37


In [18]:
# EXP007: Decision Tree min_samples_leaf = 5

model = DecisionTreeClassifier(
    min_samples_leaf=5,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.9905663171613056
Precision: 0.9883010529052385
Recall: 0.9708229585647705
F1: 0.9794840413539888
Tree depth: 38


In [19]:
# EXP008: Decision Tree min_samples_leaf = 20

model = DecisionTreeClassifier(
    min_samples_leaf=20,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.9906255553580949
Precision: 0.9903432076210361
Recall: 0.9690353061354785
F1: 0.9795733970118429
Tree depth: 36


In [20]:
# EXP009: Decision Tree với class_weight="balanced"

model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.97914815473017
Precision: 0.93842652395891
Recall: 0.9740151950456489
F1: 0.9558897243107769
Tree depth: 35


In [21]:
# EXP010: Chỉ sử dụng nhóm Length

FEATURES_LENGTH = [
    "url_length",
    "hostname_length"
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_LENGTH],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_LENGTH]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.8172501629050412
Precision: 0.6991489871748772
Recall: 0.3724063078592862
F1: 0.4859618428726152
Tree depth: 30


In [22]:
# EXP011: Không sử dụng nhóm Length

FEATURES_NO_LENGTH = [
    feature for feature in FEATURES
    if feature not in ["url_length", "hostname_length"]
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_NO_LENGTH],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_NO_LENGTH]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Features:", FEATURES_NO_LENGTH)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Features: ['dot_count', 'hyphen_count', 'digit_count', 'at_count', 'slash_count', 'question_count', 'equal_count', 'has_https']
Accuracy: 0.9880931224453527
Precision: 0.9807804309842749
Recall: 0.967630722083892
F1: 0.974161203239491
Tree depth: 28


In [23]:
# EXP012: Chỉ sử dụng nhóm Host Shape

FEATURES_HOST_SHAPE = [
    "dot_count",
    "hyphen_count"
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_HOST_SHAPE],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_HOST_SHAPE]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.8407084888335999
Precision: 0.8936306754371891
Recall: 0.35561514396986527
F1: 0.5087687248812568
Tree depth: 16


In [24]:
# EXP013: Không sử dụng nhóm Host Shape

FEATURES_NO_HOST_SHAPE = [
    feature for feature in FEATURES
    if feature not in ["dot_count", "hyphen_count"]
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_NO_HOST_SHAPE],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_NO_HOST_SHAPE]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Features:", FEATURES_NO_HOST_SHAPE)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Features: ['url_length', 'hostname_length', 'digit_count', 'at_count', 'slash_count', 'question_count', 'equal_count', 'has_https']
Accuracy: 0.983694686333748
Precision: 0.9657753326509724
Recall: 0.9638638830364553
F1: 0.9648186611279758
Tree depth: 36


In [25]:
# EXP014: Chỉ sử dụng nhóm Special Characters

FEATURES_SPECIAL_CHARS = [
    "digit_count",
    "at_count",
    "slash_count",
    "question_count",
    "equal_count"
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_SPECIAL_CHARS],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_SPECIAL_CHARS]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())


Accuracy: 0.8147325395414964
Precision: 0.6750305318085933
Recall: 0.388175956074826
F1: 0.4929063640048642
Tree depth: 20


In [26]:
# EXP015: Không sử dụng nhóm Special Characters

FEATURES_NO_SPECIAL_CHARS = [
    feature for feature in FEATURES
    if feature not in [
        "digit_count",
        "at_count",
        "slash_count",
        "question_count",
        "equal_count"
    ]
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_NO_SPECIAL_CHARS],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_NO_SPECIAL_CHARS]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Features:", FEATURES_NO_SPECIAL_CHARS)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Features: ['url_length', 'hostname_length', 'dot_count', 'hyphen_count', 'has_https']
Accuracy: 0.9873822640838813
Precision: 0.9784223787066348
Recall: 0.9669284300580987
F1: 0.9726414488472159
Tree depth: 32


In [27]:
# EXP016: Chỉ sử dụng đặc trưng Protocol

FEATURES_PROTOCOL = [
    "has_https"
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_PROTOCOL],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_PROTOCOL]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Accuracy: 0.987870979207393
Precision: 0.9995288733342307
Recall: 0.9481580795505331
F1: 0.9731660168408637
Tree depth: 1


In [28]:
# EXP017: Không sử dụng Protocol

FEATURES_NO_PROTOCOL = [
    feature for feature in FEATURES
    if feature != "has_https"
]

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    x_train[FEATURES_NO_PROTOCOL],
    y_train
)

y_pred = model.predict(
    x_val[FEATURES_NO_PROTOCOL]
)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Features:", FEATURES_NO_PROTOCOL)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Tree depth:", model.get_depth())

Features: ['url_length', 'hostname_length', 'dot_count', 'hyphen_count', 'digit_count', 'at_count', 'slash_count', 'question_count', 'equal_count']
Accuracy: 0.8576950417629288
Precision: 0.6798787734727835
Recall: 0.7304475515546192
F1: 0.7042565633560063
Tree depth: 39


In [29]:
from sklearn.ensemble import RandomForestClassifier
# EXP018: Random Forest

model = RandomForestClassifier(
    n_estimators=120,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Huấn luyện trên toàn bộ 10 đặc trưng
model.fit(x_train, y_train)

# Dự đoán trên Validation
y_pred = model.predict(x_val)

# Tính các chỉ số đánh giá
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Number of trees:", len(model.estimators_))

Accuracy: 0.992284224868195
Precision: 0.9960686672782073
Recall: 0.9705675796463002
F1: 0.9831527890056588
Number of trees: 120


In [30]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
# EXP019: Linear SVM

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(
        random_state=42,
        max_iter=5000,
        dual="auto"
    ))
])

# Huấn luyện
model.fit(x_train, y_train)

# Dự đoán
y_pred = model.predict(x_val)

# Đánh giá
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, pos_label="phishing")
recall = recall_score(y_val, y_pred, pos_label="phishing")
f1 = f1_score(y_val, y_pred, pos_label="phishing")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 0.9884189325276939
Precision: 0.9985259631490787
Recall: 0.9514780054906468
F1: 0.9744344187262979


In [31]:
## 20. Error Analysis - Phân tích lỗi trên tập Test

In [32]:
# Dùng lại test_df và X_test đã tạo ở các ô đầu notebook
X_test = X_test[FEATURES]
y_test = test_df[TARGET_COLUMN]

# Nạp mô hình đã chọn để đánh giá cuối trên Test
model = joblib.load(ROOT / "artifacts" / "selected_model.joblib")
model_name = type(model).__name__
y_pred = model.predict(X_test)

print("Số dòng Test:", len(test_df))
print("Số đặc trưng:", len(FEATURES))
print("Mô hình:", model_name)
print("Số dự đoán:", len(y_pred))

# Giải phóng mô hình lớn trước các bước phân tích lỗi
del model


Số dòng Test: 67524
Số đặc trưng: 10
Mô hình: RandomForestClassifier
Số dự đoán: 67524


In [33]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["legitimate", "phishing"]
)

tn, fp, fn, tp = cm.ravel()

print("CONFUSION MATRIX")
print(pd.DataFrame(
    cm,
    index=["Thực tế legitimate", "Thực tế phishing"],
    columns=["Dự đoán legitimate", "Dự đoán phishing"]
))

print("\nCHI TIẾT:")
print("TN - Legitimate dự đoán đúng:", tn)
print("FP - Legitimate bị dự đoán nhầm thành phishing:", fp)
print("FN - Phishing bị dự đoán nhầm thành legitimate:", fn)
print("TP - Phishing dự đoán đúng:", tp)

CONFUSION MATRIX
                    Dự đoán legitimate  Dự đoán phishing
Thực tế legitimate               51797                61
Thực tế phishing                  1450             14216

CHI TIẾT:
TN - Legitimate dự đoán đúng: 51797
FP - Legitimate bị dự đoán nhầm thành phishing: 61
FN - Phishing bị dự đoán nhầm thành legitimate: 1450
TP - Phishing dự đoán đúng: 14216


In [34]:
# Tạo bảng chứa kết quả thực tế và dự đoán
error_df = test_df.copy()

error_df["predicted"] = y_pred

# Lọc website phishing bị dự đoán thành legitimate (FN)
false_negative = error_df[
    (error_df["type"] == "phishing") &
    (error_df["predicted"] == "legitimate")
].copy()

# Lọc website legitimate bị dự đoán thành phishing (FP)
false_positive = error_df[
    (error_df["type"] == "legitimate") &
    (error_df["predicted"] == "phishing")
].copy()

print("Số FN:", len(false_negative))
print("Số FP:", len(false_positive))

print("\nMột số URL phishing bị bỏ sót:")
display(
    false_negative[
        ["url", "normalized_url", "hostname", "type", "predicted"]
    ].head(10)
)

print("\nMột số URL legitimate bị nhận nhầm:")
display(
    false_positive[
        ["url", "normalized_url", "hostname", "type", "predicted"]
    ].head(10)
)

Số FN: 1450
Số FP: 61

Một số URL phishing bị bỏ sót:


,url,normalized_url,hostname,type,predicted
51867,https://www.paklap.pk/,https://www.paklap.pk/,www.paklap.pk,phishing,legitimate
51868,https://www.paklap.pk/portal-brasil/,https://www.paklap.pk/portal-brasil/,www.paklap.pk,phishing,legitimate
51871,https://groffassociates.com/docs/dokin/plswt.php,https://groffassociates.com/docs/dokin/plswt.php,groffassociates.com,phishing,legitimate
51872,https://sites.google.com/site/info230251400nt/,https://sites.google.com/site/info230251400nt/,sites.google.com,phishing,legitimate
51874,https://sites.google.com/site/info02365471000net/,https://sites.google.com/site/info02365471000net/,sites.google.com,phishing,legitimate
51875,https://sites.google.com/site/help411452211010...,https://sites.google.com/site/help411452211010...,sites.google.com,phishing,legitimate
51884,https://g2rburn.com/dwn/dwn/index.php,https://g2rburn.com/dwn/dwn/index.php,g2rburn.com,phishing,legitimate
51887,https://chinabitmain.com/,https://chinabitmain.com/,chinabitmain.com,phishing,legitimate
51888,https://sites.google.com/site/helpprotect01/,https://sites.google.com/site/helpprotect01/,sites.google.com,phishing,legitimate
51889,https://suport-registeryaccount.ga/?facebook.c...,https://suport-registeryaccount.ga/?facebook.c...,suport-registeryaccount.ga,phishing,legitimate



Một số URL legitimate bị nhận nhầm:


,url,normalized_url,hostname,type,predicted
6560,https://www.globalpovertyproject.com/blog/view...,https://www.globalpovertyproject.com/blog/view...,www.globalpovertyproject.com,legitimate,phishing
6690,https://www.greatprostatecancerchallenge.com/r...,https://www.greatprostatecancerchallenge.com/r...,www.greatprostatecancerchallenge.com,legitimate,phishing
8766,https://www.ki.se/ki/jsp/polopoly.jsp?l=en&d=8...,https://www.ki.se/ki/jsp/polopoly.jsp?l=en&d=8...,www.ki.se,legitimate,phishing
9214,https://www.limarestaurantguide.com/65506,https://www.limarestaurantguide.com/65506,www.limarestaurantguide.com,legitimate,phishing
15538,https://www.titancast.titantv.com/p/klax/v/Cam...,https://www.titancast.titantv.com/p/klax/v/Cam...,www.titancast.titantv.com,legitimate,phishing
19650,https://www.americansongspace.com/1c293b8d9bfe...,https://www.americansongspace.com/1c293b8d9bfe...,www.americansongspace.com,legitimate,phishing
20795,https://www.artandpopularculture.com/1620s,https://www.artandpopularculture.com/1620s,www.artandpopularculture.com,legitimate,phishing
20797,https://www.artandpopularculture.com/1894,https://www.artandpopularculture.com/1894,www.artandpopularculture.com,legitimate,phishing
20798,https://www.artandpopularculture.com/1896,https://www.artandpopularculture.com/1896,www.artandpopularculture.com,legitimate,phishing
20799,https://www.artandpopularculture.com/1960,https://www.artandpopularculture.com/1960,www.artandpopularculture.com,legitimate,phishing


In [35]:
print("Số lượng False Negative:", len(false_negative))

display(
    false_negative[
        ["url", "normalized_url", "hostname", "type", "predicted"]
    ].head(20)
)

Số lượng False Negative: 1450


,url,normalized_url,hostname,type,predicted
51867,https://www.paklap.pk/,https://www.paklap.pk/,www.paklap.pk,phishing,legitimate
51868,https://www.paklap.pk/portal-brasil/,https://www.paklap.pk/portal-brasil/,www.paklap.pk,phishing,legitimate
51871,https://groffassociates.com/docs/dokin/plswt.php,https://groffassociates.com/docs/dokin/plswt.php,groffassociates.com,phishing,legitimate
51872,https://sites.google.com/site/info230251400nt/,https://sites.google.com/site/info230251400nt/,sites.google.com,phishing,legitimate
51874,https://sites.google.com/site/info02365471000net/,https://sites.google.com/site/info02365471000net/,sites.google.com,phishing,legitimate
51875,https://sites.google.com/site/help411452211010...,https://sites.google.com/site/help411452211010...,sites.google.com,phishing,legitimate
51884,https://g2rburn.com/dwn/dwn/index.php,https://g2rburn.com/dwn/dwn/index.php,g2rburn.com,phishing,legitimate
51887,https://chinabitmain.com/,https://chinabitmain.com/,chinabitmain.com,phishing,legitimate
51888,https://sites.google.com/site/helpprotect01/,https://sites.google.com/site/helpprotect01/,sites.google.com,phishing,legitimate
51889,https://suport-registeryaccount.ga/?facebook.c...,https://suport-registeryaccount.ga/?facebook.c...,suport-registeryaccount.ga,phishing,legitimate


In [36]:
# Tạo bảng phân tích lỗi
analysis_df = test_df[
    ["url", "normalized_url", "hostname", "type"]
].copy()

analysis_df["predicted"] = y_pred
analysis_df["has_https"] = X_test["has_https"].values

# Chia các nhóm
fn = analysis_df[
    (analysis_df["type"] == "phishing") &
    (analysis_df["predicted"] == "legitimate")
]

tp = analysis_df[
    (analysis_df["type"] == "phishing") &
    (analysis_df["predicted"] == "phishing")
]

print("=== HTTPS TRONG CÁC URL PHISHING ===")

print("FN dùng HTTPS:", fn["has_https"].mean() * 100, "%")
print("TP dùng HTTPS:", tp["has_https"].mean() * 100, "%")

print("\n=== HOSTNAME XUẤT HIỆN NHIỀU TRONG FN ===")

display(
    fn["hostname"]
    .value_counts()
    .head(15)
    .rename("number_of_false_negatives")
    .to_frame()
)

print("\n=== THỐNG KÊ ĐỘ DÀI URL ===")

print("Độ dài URL trung bình của FN:",
      fn["normalized_url"].str.len().mean())

print("Độ dài URL trung bình của TP:",
      tp["normalized_url"].str.len().mean())

=== HTTPS TRONG CÁC URL PHISHING ===
FN dùng HTTPS: 100.0 %
TP dùng HTTPS: 2.25801913337085 %

=== HOSTNAME XUẤT HIỆN NHIỀU TRONG FN ===


,number_of_false_negatives
hostname,
sites.google.com,971
www.google.com,24
onedrive.live.com,21
plus.google.com,17
i.imgur.com,8
www.vakantieverhuur.be,6
saridity.com,5
jimryky.blogspot.com,5
infoactive.000webhostapp.com,4



=== THỐNG KÊ ĐỘ DÀI URL ===
Độ dài URL trung bình của FN: 57.677241379310345
Độ dài URL trung bình của TP: 62.02061057962859


In [37]:
def https_statistics(frame, split_name):
    url = frame["normalized_url"].fillna("").astype(str)

    temp = pd.DataFrame({
        "type": frame["type"],
        "has_https": url.str.lower().str.startswith("https://")
    })

    result = (
        temp.groupby("type")["has_https"]
        .agg(
            total="count",
            https_count="sum",
            https_rate="mean"
        )
    )

    result["https_rate"] = (
        result["https_rate"] * 100
    ).round(2)

    print(f"\n===== {split_name} =====")
    display(result)


https_statistics(train, "TRAIN")
https_statistics(val, "VALIDATION")
https_statistics(test_df, "TEST")


===== TRAIN =====


,total,https_count,https_rate
type,,,
legitimate,242019,241997,99.99
phishing,73106,3900,5.33



===== VALIDATION =====


,total,https_count,https_rate
type,,,
legitimate,51861,51854,99.99
phishing,15663,812,5.18



===== TEST =====


,total,https_count,https_rate
type,,,
legitimate,51858,51851,99.99
phishing,15666,1771,11.30


In [3]:
# EXP021: Random Forest bỏ đặc trưng has_https
FEATURES_NO_HTTPS = [
    feature for feature in FEATURES
    if feature != "has_https"
]

model_rf_no_https = RandomForestClassifier(
    n_estimators=120,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model_rf_no_https.fit(
    x_train[FEATURES_NO_HTTPS],
    y_train
)

train_pred = model_rf_no_https.predict(x_train[FEATURES_NO_HTTPS])
val_pred = model_rf_no_https.predict(x_val[FEATURES_NO_HTTPS])

print("Số đặc trưng:", len(FEATURES_NO_HTTPS))
print("Train accuracy:", accuracy_score(y_train, train_pred))
print("Validation accuracy:", accuracy_score(y_val, val_pred))
print("Validation precision:", precision_score(
    y_val, val_pred, pos_label="phishing", zero_division=0
))
print("Validation recall:", recall_score(
    y_val, val_pred, pos_label="phishing", zero_division=0
))
print("Validation F1:", f1_score(
    y_val, val_pred, pos_label="phishing", zero_division=0
))

NameError: name 'FEATURES' is not defined